In [1]:
# import libraries



from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2 import service_account
from oauth2client.service_account import ServiceAccountCredentials 
import gspread


# Standard libraries
import os  # file path operations
import urllib.request  # downloading the BOT-SORT config
from datetime import datetime  

import cv2  # image processing
import numpy as np  # numerical operations

from collections import defaultdict  # dictionary with default types

from ultralytics import YOLO  # object detection
import easyocr  # OCR

# Google API client libraries
from googleapiclient.discovery import build  # interacting with Google APIs
from googleapiclient.http import MediaFileUpload  #  uploading files
from google.oauth2 import service_account  #  authentication (preferred modern method)
from oauth2client.service_account import ServiceAccountCredentials  #  older Google API auth (consider removing if unused)

import gspread  # interacting with Google Sheets

In [2]:
video_path         = '/Users/masaaladwan/Downloads/IMG_8827.MOV'
output_dir         = '/Users/masaaladwan/Desktop/Obj /Final_Connected'
unsafe_dir         = os.path.join(output_dir, '/Users/masaaladwan/Desktop/Obj /Final_Connected/violation')
rotation_angle     = 0        # 0 = no rotation; ±90 = rotate
lane_polygons_path = '/Users/masaaladwan/Desktop/Obj /lane_polygons_2.npy' # manual lane detection 

#filtering out low confidence or too-small (far away, false) detections 
conf_thresh        = 0.4     # min YOLO confidence
area_thresh        = 500     # min box area (px²)

# for Hough lane detection 
min_lane_len       = 100  # its not used here, I did the lane detection manually, cuz i didnt know how to do it Automatically  
lane_merge_th      = 40   # will write comment later , i forgot what it does 💅 

# Safe distance thresholds 
safe_base          = 120  # base pixel gap at mid-range
band_ratios        = [0.6, 1.0, 1.7]  # multipliers for [far, mid, near]


pad_ratio          = 0.1 # crop padding 

smoothing_steps    = 2  #Helps Avoid FP from glitch 

# everything below this is considered to be the Street 
HORIZON_OVERRIDE   = 688     



lane_fill_colors = [(0, 255, 0), (255, 0, 0), (0, 255, 255)]
edge_colors = [(0, 255, 0), (0, 0, 255), (255, 0, 0), (0, 255, 255)]


In [3]:
# ensure BOT-SORT config is present
if not os.path.exists('botsort.yaml'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/trackers/botsort.yaml',
        'botsort.yaml'
    )

In [4]:
lane_edges = np.load(lane_polygons_path, allow_pickle=True)

# making sure there is 4 edges for 3 lanes, for mycase 
if len(lane_edges) != 4:
    raise ValueError("Expected 4 lane edges for 3 lanes")



lane_polygons = []
for i in range(3):
    poly = np.vstack([lane_edges[i], lane_edges[i+1][::-1]])
    lane_polygons.append(poly)

In [5]:
os.makedirs(output_dir, exist_ok=True)
os.makedirs(unsafe_dir, exist_ok=True)

In [6]:
# Oprn the video and get the fps 
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
ret, sample = cap.read()
cap.release() # to free up memory 


if not ret:
    raise RuntimeError("Cannot read video sample")

H, W = sample.shape[:2] 
fps_int = int(round(fps))

horizon_y = HORIZON_OVERRIDE or 0

In [7]:
fps


29.994056659842833

In [8]:
model = YOLO('yolo11n.pt')
stream = model.track(source=video_path, tracker='botsort.yaml', persist=True, stream=True)

# Sets up to write annotated frames to a new video file
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(os.path.join(output_dir, 'annotated_output.mp4'), fourcc, fps, (W, H))
if not out.isOpened():
    raise RuntimeError("Failed to open video writer")

In [9]:

def get_safe_thresh(cy, horizon, H):
    band_h = (H - horizon) / 3 # each one will have different threshold 

    rel = (cy - horizon) / band_h # to know which band the car is in 

    if rel < 1: return safe_base * band_ratios[0]

    elif rel < 2: return safe_base * band_ratios[1]
    
    else: return safe_base * band_ratios[2]

In [10]:
violation_log_entries = []  

frames_per_second = fps_int
desired_frames_per_second = 5  # can be 2, 3, 4, or 5
sampling_interval = max(1, frames_per_second // desired_frames_per_second)

In [1]:
# === not used later ===
# Speed Estmamtion to solve the taxi problem
""" Logic : 
Estimate speed from position deltas:
	•	If front car is moving slowly or not at all
	•	And rear car is moving fast → probably overtaking """ 


def estimate_speed(tid, position_history, max_history=5):
  
    history = position_history.get(tid, [])
    print(history)
    if len(history) < 2:
        return 0.0  # Not enough data to estimate speed

    # Use only the first and last entries for delta calculation
    (f_start, y_start), (f_end, y_end) = history[0], history[-1]
    frame_diff = f_end - f_start

    if frame_diff == 0:
        return 0.0  # Avoid division by zero

    return (y_end - y_start) / frame_diff  # Pixels per frame 

In [12]:
# === not used later ===

def crop_plate_region(car_img):
    h, w = car_img.shape[:2]
    y1 = int(h * 0.75)
    x1 = int(w * 0.25)
    x2 = int(w * 0.75)
    return car_img[y1:h, x1:x2]

In [13]:
def clean_plate_digits(text):
    digits = ''.join(c for c in text if c.isdigit())
    return digits[-7:] if len(digits) >= 7 else digits



In this code, I used two different logics for grouping detected text into rows of the plate:

1️. The first logic groups text by comparing the vertical center (y-center) of each detected box with existing lines. If the y-center is close enough to an existing line’s center, it’s added to that line. This approach is more flexible and works better for plates that are skewed, tilted, or not perfectly horizontal.

2️. The second logic simply checks if the y-coordinate of a new text box is close to the previous one. If so, it groups them in the same line; otherwise, it starts a new line. This method is simpler and faster, and it works well when the plate rows are cleanly aligned.

In short:
	•	The first logic is more robust for messy or angled plates.
	•	The second logic is lighter and good for clean, straight plates.



In [14]:

# === EasyOCR setup ===
reader = easyocr.Reader(['en'])

# Fix common misreads 
def fix_common_plate_errors(text):
    text = text.upper()
    return (text.replace('S', '5')
                .replace('B', '8')
                .replace('Z', '2')
                .replace('O', '0')
                .replace('I', '1'))

# OCR function 
def recognize_plate_2rows(image_path):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"❌ File not found: {image_path}")

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    sharpen = cv2.filter2D(contrast, -1, kernel=np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))

    results = reader.readtext(sharpen, detail=1)
    if not results:
        return None

    sorted_results = sorted(results, key=lambda r: r[0][0][1])

    lines = []
    current_line = []
    last_y = None
    threshold = 15

    for box, text, conf in sorted_results:
        y = box[0][1]
        if last_y is None or abs(y - last_y) < threshold:
            current_line.append(text.strip())
        else:
            lines.append(current_line)
            current_line = [text.strip()]
        last_y = y

    if current_line:
        lines.append(current_line)

    combined = [''.join(fix_common_plate_errors(t) for t in line) for line in lines]
    only_digits = ''.join(c for line in combined for c in line if c.isdigit())
    return only_digits

def clean_plate_digits(digits):
    digits = ''.join(filter(str.isdigit, digits))
    if len(digits) > 7:
        print(f"⚠️ Warning: OCR returned {len(digits)} digits. Cropping to last 7.")
        return digits[-7:]  # Keep last 7
    elif len(digits) < 7:
        print(f"⚠️ Warning: OCR returned only {len(digits)} digits. Might be incomplete.")
    return digits


In [15]:
def recognize_plate_2rows(image_path):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"❌ File not found: {image_path}")

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    sharpen = cv2.filter2D(contrast, -1, kernel=np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))

    results = reader.readtext(sharpen, detail=1)
    if not results:
        return None

    # Sort by y-center of bounding box
    results = sorted(results, key=lambda r: np.mean([r[0][0][1], r[0][2][1]]))

    lines = []
    current_line = []
    line_tops = []

    for box, text, conf in results:
        y_top = min([pt[1] for pt in box])
        y_bottom = max([pt[1] for pt in box])
        y_center = (y_top + y_bottom) / 2

        matched = False
        for i, existing_center in enumerate(line_tops):
            if abs(y_center - existing_center) < 25:  # more lenient now
                lines[i].append(text.strip())
                line_tops[i] = (line_tops[i] + y_center) / 2  # average update
                matched = True
                break
        if not matched:
            lines.append([text.strip()])
            line_tops.append(y_center)

    combined = [''.join(fix_common_plate_errors(t) for t in line) for line in lines]
    only_digits = ''.join(c for line in combined for c in line if c.isdigit())
    return only_digits

In [16]:


reader = easyocr.Reader(['en'])

def fix_common_plate_errors(text):
    text = text.upper()
    return (text.replace('S', '5')
                .replace('B', '8')
                .replace('Z', '2')
                .replace('O', '0')
                .replace('I', '1'))

def recognize_plate_2rows(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    sharpen = cv2.filter2D(contrast, -1, kernel=np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]]))

    results = reader.readtext(sharpen, detail=1)

    if not results:
        return None

    # Sort results top-to-bottom
    sorted_results = sorted(results, key=lambda r: r[0][0][1])

    # Try to detect row clusters (Y-distance threshold)
    lines = []
    current_line = []
    last_y = None
    threshold = 15  # pixel distance between rows

    for box, text, conf in sorted_results:
        y = box[0][1]
        if last_y is None or abs(y - last_y) < threshold:
            current_line.append(text.strip())
        else:
            lines.append(current_line)
            current_line = [text.strip()]
        last_y = y

    if current_line:
        lines.append(current_line)

    # Join each row
    combined = [''.join(fix_common_plate_errors(t) for t in line) for line in lines]
    all_text = ''.join(combined)

    # Extract only digits
    digits = ''.join(c for c in all_text if c.isdigit())

    if len(digits) == 7:
        return digits
    elif len(digits) > 7:
        print(f"⚠️ OCR returned {len(digits)} digits. Cropping to last 7.")
        return digits[-7:]
    else:
        print(f"❌ OCR returned only {len(digits)} digits. Skipping.")
        return None

In [17]:
img = cv2.imread("/Users/masaaladwan/Desktop/Obj /plate_processed.jpg")
plate = recognize_plate_2rows(img)
print("Detected Plate:", plate)

⚠️ OCR returned 8 digits. Cropping to last 7.
Detected Plate: 0127163


In [19]:
def crop_bottom_half(image):
    h, w = image.shape[:2]
    return image[h//2 : h, 0 : w]

# Google

In [20]:
# === Setup ===
SERVICE_ACCOUNT_FILE = '/Users/masaaladwan/Desktop/Obj /Jun/cars-462612-039e7e1353b9.json'
SCOPES = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
FOLDER_ID = '1suzow11KrdX9qKA7vUcG4UlQGtWG797I'
SHEET_NAME = 'Data'  # Must match your actual sheet name

# === Google Sheets Auth ===
creds_sheet = ServiceAccountCredentials.from_json_keyfile_name(SERVICE_ACCOUNT_FILE, SCOPES)
client = gspread.authorize(creds_sheet)
sheet = client.open(SHEET_NAME).sheet1

# === Google Drive Auth ===
creds_drive = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
drive_service = build('drive', 'v3', credentials=creds_drive)

# === Upload function ===
def upload_to_drive(filepath, filename):
    file_metadata = {
        'name': filename,
        'parents': [FOLDER_ID]
    }
    media = MediaFileUpload(filepath, resumable=True)
    file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id,webViewLink'
    ).execute()
    return file.get('webViewLink')

# === Logging Function ===
def log_violation(plate_number, timestamp, image_url):
    sheet.append_row([plate_number, timestamp, image_url])
    print(f"📝 Logged: {plate_number}, {timestamp}, {image_url}")

# Main Loop 

In [21]:
# === Initial Setup ===
violation_counts = {}
saved_ids = set()
pending_violators = set()
track_pending_close_capture = {}  # track_id → (max_cy, bbox)
last_seen_frame = {}
frame_idx = 0
prev_positions = {}
car_speeds = {}
speed_history = defaultdict(list)
lane_history = defaultdict(list)
speed_threshold = 0.2
lane_history_window = 10

# Main Loop 
for r in stream:
    frame = r.orig_img.copy()

    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)
    ids = r.boxes.id.cpu().numpy().astype(int)

    cars = []
    for (x1, y1, x2, y2), conf, cls, tid in zip(boxes, confs, classes, ids):
        if cls != 2 or conf < conf_thresh:
            continue
        if (x2 - x1) * (y2 - y1) < area_thresh:
            continue

        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if cy < horizon_y:
            continue

        front_x, front_y = int((x1 + x2) / 2), int(y2)
        cars.append((tid, cx, cy, x1, y1, x2, y2, front_x, front_y))
        last_seen_frame[tid] = frame_idx

    # Assign cars to lanes
    cars_by_lane = [[] for _ in lane_polygons]
    car_lane_assignment = {}

    for car in cars:
        tid, cx, cy, x1, y1, x2, y2, front_x, front_y = car
        for lane_idx, poly in enumerate(lane_polygons):
            if cv2.pointPolygonTest(poly.astype(np.int32), (cx, cy), False) >= 0:
                cars_by_lane[lane_idx].append(car)
                car_lane_assignment[tid] = lane_idx + 1
                lane_history[tid].append(lane_idx + 1)
                if len(lane_history[tid]) > lane_history_window:
                    lane_history[tid].pop(0)
                break

    for car in cars:
        tid, cx, cy, *_ = car
        current_pos = np.array([cx, cy])
        if tid in prev_positions:
            prev_pos = prev_positions[tid]
            delta = np.linalg.norm(current_pos - prev_pos)
            if delta > 0.01:
                car_speeds[tid] = delta
                speed_history[tid].append(delta)
                if len(speed_history[tid]) > 10:
                    speed_history[tid].pop(0)
        prev_positions[tid] = current_pos

    def consistently_in_lane(tid, lane_id):
        history = lane_history.get(tid, [])
        if len(history) < 3:
            return True
        return history.count(lane_id) / len(history) >= 0.7

    raw_violators = set()
    violation_counts = {}

    for lane_idx, lane_cars in enumerate(cars_by_lane):
        if len(lane_cars) < 2:
            continue

        sorted_lane = sorted(lane_cars, key=lambda c: c[2], reverse=True)

        for i in range(len(sorted_lane) - 1):
            front = sorted_lane[i]
            rear = sorted_lane[i + 1]

            tid_f, cx_f, cy_f, *_ = front
            tid_r, cx_r, cy_r, *_ = rear

            if abs(cx_f - cx_r) > 50:
                continue

            if not consistently_in_lane(tid_f, lane_idx + 1) or not consistently_in_lane(tid_r, lane_idx + 1):
                continue

            speed_f = car_speeds.get(tid_f, 0)
            speed_r = car_speeds.get(tid_r, 0)
            if speed_f < 1.0 and speed_r > speed_f + 1.5:
                continue

            gap = cy_f - cy_r
            thresh = get_safe_thresh(cy_r, horizon_y, H)

            if 0 < gap < thresh:
                raw_violators.add(tid_r)
                violation_counts[tid_r] = violation_counts.get(tid_r, 0) + 1
                pending_violators.add(tid_r)
            else:
                violation_counts[tid_r] = 0

    smooth_violators = {
        tid for tid, count in violation_counts.items() if count >= 1
    }

    # Track violator position for closest capture
    for car in cars:
        tid, cx, cy, x1, y1, x2, y2, *_ = car
        if tid in pending_violators:
            if tid not in track_pending_close_capture or cy > track_pending_close_capture[tid][0]:
                track_pending_close_capture[tid] = (cy, (x1, y1, x2, y2))

    # Save snapshot only when violator is close (Band 1)
    band_h = (H - horizon_y) / 3
    for tid, (cy, (x1, y1, x2, y2)) in track_pending_close_capture.items():
        if tid in saved_ids:
            continue
        if cy > horizon_y + 2 * band_h:
            saved_ids.add(tid)
            w, h_ = x2 - x1, y2 - y1
            px, py = int(w * pad_ratio), int(h_ * pad_ratio)
            x0, y0 = max(0, int(x1)-px), max(0, int(y1)-py)
            x1p, y1p = min(W, int(x2)+px), min(H, int(y2)+py)
            crop = frame[y0:y1p, x0:x1p]
            if crop.size:
                filename = f"track{tid}_close.jpg"
                cv2.imwrite(os.path.join(unsafe_dir, filename), crop)

                # === 1. Crop the plate from bottom of car image
                plate_img = crop_bottom_half(crop)
                plate_path = os.path.join(unsafe_dir, f"track{tid}_plate.jpg")
                cv2.imwrite(plate_path, plate_img)

                # === 2. OCR
                plate_img = cv2.imread(plate_path)
                plate_raw = recognize_plate_2rows(plate_img)
                if not plate_raw:
                    print(f"❌ No plate detected for TID {tid}")
                    continue  # skip this one

                plate_clean = clean_plate_digits(plate_raw)
                plate_clean = clean_plate_digits(plate_raw)
                print(f"✅ Plate Detected (TID {tid}):", plate_clean)

                # === 3. Upload to Drive
                image_url = upload_to_drive(os.path.join(unsafe_dir, filename), filename)

                # === 4. Log to Google Sheet
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                log_violation(plate_clean, timestamp, image_url)
                print(f"📸 Saved close-up for TID {tid} as {filename}")



    # Draw the lane 
    annotated = frame.copy()
    overlay = annotated.copy()

    for lane_idx, poly in enumerate(lane_polygons):
        cv2.fillPoly(overlay, [poly.astype(np.int32).reshape(-1,1,2)], lane_fill_colors[lane_idx])
        cx = int(np.mean(poly[:,0]))
        cy = int(np.mean(poly[:,1]))
        cv2.putText(overlay, f"Lane {lane_idx+1}", (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    # Band lines
    band_h = (H - horizon_y) / 3
    for i in range(4):
        y = int(horizon_y + i * band_h)
        cv2.line(overlay, (0, y), (W, y), (200, 200, 200), 1)
        if i < 3:
            cv2.putText(overlay, f"Band {i+1}", (10, y + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100,100,100), 1)

    annotated = cv2.addWeighted(annotated, 1, overlay, 0.3, 0)

    # Draw lane edges
    for idx, edge in enumerate(lane_edges):
        cv2.polylines(annotated, [edge.astype(np.int32).reshape(-1,1,2)], isClosed=False, color=edge_colors[idx % len(edge_colors)], thickness=2)

    # Draw car boxes
    for (tid, cx, cy, x1, y1, x2, y2, front_x, front_y) in cars:
        pt1, pt2 = (int(x1), int(y1)), (int(x2), int(y2))
        color = (0,0,255) if tid in smooth_violators else (255,0,0)
        cv2.rectangle(annotated, pt1, pt2, color, 2)
        lane_id = car_lane_assignment.get(tid, '?')
        cv2.putText(annotated, f"ID{tid} L{lane_id}", (pt1[0], pt1[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # Save violation snapshot, only once per second 
        if (frame_idx % fps_int == fps_int - 1 and tid in smooth_violators and tid not in saved_ids):
            
            w, h_ = x2 - x1, y2 - y1
            px, py = int(w * pad_ratio), int(h_ * pad_ratio)
            x0, y0 = max(0, int(x1)-px), max(0, int(y1)-py)
            x1p, y1p = min(W, int(x2)+px), min(H, int(y2)+py)
            crop = frame[y0:y1p, x0:x1p]
            if crop.size:
                cv2.imwrite(os.path.join(unsafe_dir, f"track{tid}.jpg"), crop)

    # Save the annotated frame to the video 
    out.write(annotated)
    
    # Violation logging 
    if frame_idx % sampling_interval == 0:
        for lane_idx, lane_cars in enumerate(cars_by_lane):
            if len(lane_cars) < 2:
                continue

            sorted_cars = sorted(lane_cars, key=lambda c: c[2])  # sort by cy (N→S)

            for i in range(len(sorted_cars) - 1):
                front = sorted_cars[i]
                rear = sorted_cars[i + 1]

                tid_f, _, cy_f, *_ = front
                tid_r, cx_r, cy_r, *_ = rear
                gap = front[6] - rear[8]
                thresh = get_safe_thresh(cy_r, horizon_y, H)
                violation = 0 < gap < thresh

                violation_log_entries.append({
                    'frame': frame_idx,
                    'timestamp': frame_idx / fps_int,
                    'car_id': tid_r,
                    'front_car_id': tid_f,
                    'lane': lane_idx + 1,
                    'cx': cx_r,
                    'cy': cy_r,
                    'front_cx': front_x,
                    'front_cy': front_y,
                    'distance_to_next_car': gap,
                    'violation': violation
                })


    frame_idx += 1

out.release()
print("Done — saved to:", os.path.join(output_dir, 'annotated_output.mp4'))


video 1/1 (frame 1/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 16 cars, 36.7ms
video 1/1 (frame 2/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 15 cars, 36.3ms
video 1/1 (frame 3/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 15 cars, 34.2ms
video 1/1 (frame 4/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 16 cars, 36.2ms
video 1/1 (frame 5/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 16 cars, 34.8ms
video 1/1 (frame 6/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 16 cars, 34.7ms
video 1/1 (frame 7/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 17 cars, 34.0ms
video 1/1 (frame 8/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 17 cars, 33.1ms
video 1/1 (frame 9/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 17 cars, 34.0ms
video 1/1 (frame 10/757) /Users/masaaladwan/Downloads/IMG_8827.MOV: 640x384 1 person, 17 cars, 34.6ms
video 1/1 (frame 1